In [1]:
import json
import os
import pandas as pd
import yfinance as yf
from datetime import date
from scipy.stats import gmean
from yfetch import delay, get_stock_history, get_stock_name, get_stock_metadata, to_weekly

risk_free_return = 4/100
history_weeks = 3 * 52 # 3Y
change_weeks = 52 # 1Y
range_window = 4 # ~ month
cache_days = 2 # how long to cache stock data
revenue_cache_file = 'data/revenue_cache.json'

symbols = [
    'MAGS',
    'FNGS',
    'IGM',
    'SPYG',
    'SPMO',
    'IWMO.MI',
    'TSLA',
    'NVDA',
    'AVGO',
    'SMH',
    'USD',
    'TDIV.AS',
    'ESIF.DE',
    'DFEN.DE',
    'EXI',
    '4GLD.DE',
    'BRK-B',
    'NFLX',
    'PLTR',
    'NET',
    'ISRG',
    'SHOP',
    'DAPP',
    'BITQ',
    'UFO',
    'ROKT',
    'JEDI.DE',
    'SPCX',
    'RKLB',
    'PL',
    'NASA',
    'NUKZ',
    'NLR',
    'AIPO',
    'QTUM',
    'AGIX',
    'CHAT',
    'BAI',
    'ARKG',
    'PPA',
    'QQQ',
    'COIN',
    'NKE',
    'STLA',
    'AMAT',
]

try:
    with open(revenue_cache_file) as f:
        revenue_cache = json.load(f)
except FileNotFoundError:
    revenue_cache = {}

def fetch_revenue_growth(symbol):
    """(year over year growth of the latest quarterly revenue, quarter end date),
    (None, None) if unavailable."""
    delay()
    qis = yf.Ticker(symbol).quarterly_income_stmt
    if 'Total Revenue' not in qis.index:
        print(f'No revenue for {symbol}')
        return None, None
    rev = qis.loc['Total Revenue'].dropna().sort_index(ascending=False)
    if len(rev) < 5:
        print(f'Not enough revenue history for {symbol}: {len(rev)} quarters, need 5')
        return None, None
    return round(float(rev.iloc[0] / rev.iloc[4] - 1), 4), str(rev.index[0].date())

def format_revenue(growth, quarter):
    return 'n/a' if growth is None else f'{growth:.2%} ({quarter})'

def get_revenue_growth(symbol):
    """Revenue growth from data/revenue_cache.json, refetched once it goes stale."""
    if get_stock_metadata(symbol).get('instrumentType') != 'EQUITY':
        return None # ETFs and the like have no revenue

    today = date.today()
    cached = revenue_cache.get(symbol)
    if cached and (today - date.fromisoformat(cached['fetched'])).days < cache_days:
        return cached['growth']

    growth, quarter = fetch_revenue_growth(symbol)
    if cached and (growth, quarter) != (cached['growth'], cached['quarter']):
        print(f'{symbol} revenue: {format_revenue(cached["growth"], cached["quarter"])}'
              f' => {format_revenue(growth, quarter)}')

    revenue_cache[symbol] = {'growth': growth, 'quarter': quarter, 'fetched': today.isoformat()}
    os.makedirs(os.path.dirname(revenue_cache_file), exist_ok=True)
    with open(revenue_cache_file, 'w') as f:
        json.dump(revenue_cache, f, indent=2, sort_keys=True)
    return growth

def temperature(series):
    """Share of past values at or below the last one, None if the series is empty."""
    series = series.dropna()
    if len(series) == 0:
        return None
    return (series <= series.iloc[-1]).mean()

def sma_temperature(symbol, daily, window=200):
    """Temperature of the distance to the SMA over daily history."""
    sma = daily.Close.rolling(window=window).mean()
    if sma.dropna().empty:
        print(f'Not enough history for {symbol}: {len(daily)} days, need {window}')
        return None
    return temperature(daily.Close / sma - 1)

rows = []
for symbol in symbols:
    # one daily fetch per symbol - the weekly bars are aggregated from it
    daily = get_stock_history(symbol, period='5y', interval='1d', cache_days=cache_days)
    weekly = to_weekly(daily, symbol).tail(history_weeks)

    if len(weekly) < history_weeks:
        print(f'Not enough history for {symbol}: {len(weekly)} weeks, need {history_weeks}')
        gmean_change = std = sharpe = None
    else:
        changes = weekly.Close.pct_change(periods=change_weeks, fill_method=None).dropna()
        gmean_change = gmean(1 + changes) - 1 # geometric mean of changes
        std = changes.std()
        sharpe = (gmean_change - risk_free_return) / std

    high = weekly.High.rolling(range_window).max()
    low = weekly.Low.rolling(range_window).min()
    range = ((high - low) / (high + low) * 2).dropna() # relative range per window

    rows.append({
        'symbol': symbol,
        'name': get_stock_name(symbol),
        'price': weekly.Close.iloc[-1], # last close
        'weeks': len(weekly),
        'gmean': gmean_change,
        'std': std,
        'sharpe': sharpe,
        'range': range.mean(),
        'revenue': get_revenue_growth(symbol),
        'T200': sma_temperature(symbol, daily, window=200),
    })

df = pd.DataFrame(rows)
f = f'data/folio.csv'
df.to_csv(f, index=False)
print(f'Saved to {f} ({len(df)} rows)')

df = df.reset_index(drop=True)
for col in ['gmean', 'std', 'range', 'revenue', 'T200']:
    df[col] = df[col].map(lambda v: None if pd.isna(v) else f'{v:.2%}')
df

Fetched history for TSLA (1255 days)
AVGO revenue: 47.87% (2026-04-30) => 85.50% (2026-07-31)
Fetched history for TDIV.AS (1280 days)
No revenue for 4GLD.DE
Fetched history for NFLX (1255 days)
Fetched history for PLTR (1255 days)
Fetched history for NET (1255 days)
Fetched history for ISRG (1255 days)
Fetched history for SHOP (1255 days)
Fetched history for SPCX (63 days)
Not enough history for SPCX: 14 weeks, need 156
Not enough revenue history for SPCX: 4 quarters, need 5
Not enough history for SPCX: 63 days, need 200
Fetched history for RKLB (1255 days)
Fetched history for PL (1255 days)
Fetched history for NASA (114 days)
Not enough history for NASA: 24 weeks, need 156
No revenue for NASA
Not enough history for NASA: 114 days, need 200
Not enough history for NUKZ: 138 weeks, need 156
Not enough history for AIPO: 60 weeks, need 156
Not enough history for AGIX: 113 weeks, need 156
Not enough history for BAI: 99 weeks, need 156
Fetched history for ARKG (1255 days)
Fetched history for

,symbol,name,price,weeks,gmean,std,sharpe,range,revenue,T200
0,MAGS,Roundhill Magnificent Seven ETF,69.889999,156,32.88%,17.11%,1.687605,10.54%,None,30.61%
1,FNGS,MicroSectors FANG+ ETN,80.040001,156,30.37%,14.91%,1.767761,10.64%,None,49.24%
2,IGM,iShares Expanded Tech Sector ETF,162.240005,156,31.23%,14.42%,1.889055,9.92%,None,62.03%
3,SPYG,State Street SPDR Portfolio S&P 500 Growth ETF,120.620003,156,25.75%,9.63%,2.259375,7.77%,None,45.17%
4,SPMO,Invesco S&P 500 Momentum ETF,148.309998,156,33.45%,13.12%,2.244496,8.78%,None,56.06%
5,IWMO.MI,iShares Edge MSCI World Momentum Factor UCITS ...,100.690002,156,19.43%,13.85%,1.114227,7.67%,None,71.78%
6,TSLA,"Tesla, Inc.",365.440002,156,39.77%,30.49%,1.172966,23.03%,25.52%,34.19%
7,NVDA,NVIDIA Corporation,218.289993,156,59.63%,60.24%,0.923475,19.07%,105.85%,31.16%
8,AVGO,Broadcom Inc.,361.989990,156,72.61%,31.94%,2.147893,19.91%,85.50%,12.41%
9,SMH,VanEck Semiconductor ETF,568.530029,156,48.72%,43.14%,1.036540,14.93%,None,55.49%
